In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
%config InlineBackend.figure_format = "retina"
sns.set_theme(context='talk', style='ticks', palette='muted')

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# DataFrame pre-processing

In [ ]:
DATASET = "COLLIE"
SUBTASKS = "Sentence-Level Tasks"
# SUBTASKS = "Paragraph-Level Tasks"
# SUBTASKS = "Combined Tasks"

HUGGINGFACE_MODEL = "Llama-3.2-1B"

TIMESTAMP_TO_PIPELINES = {
    # VERIFAI SUBMISSION
    # baselines
    "2024-12-30-18-41-47": ["huggingface", "huggingface_beam_search", "gpt-4o-mini", "gpt-4o"], # sent
    # "2024-12-30-18-42-27": ['huggingface', 'huggingface_beam_search', 'gpt-4o-mini', 'gpt-4o'], # para
    # disciple
    "2025-01-25-16-14-55": ["disciple_smc", "disciple_iid"],  # sent
    # "2025-01-25-16-35-26": ['disciple_smc', 'disciple_iid'], # para
    # o1
    "2025-01-29-15-16-02": ["o1"], # sent
    # "2025-01-30-17-58-23": ["o1"], # para
    # DEBUGGING
    # "2025-02-19-13-24-38": ["disciple_smc", "disciple_importance"]
    # "2025-02-20-10-41-58": ["disciple_smc", "disciple_importance", "disciple_rejection"]
    # "2025-02-20-14-17-35": [
    #     "disciple_smc",
    #     "disciple_importance",
    #     "gpt-4o-mini",
    #     "gpt-4o",
    #     "o1"
    # ]
}

PIPELINES = {
    "disciple_smc": f"{HUGGINGFACE_MODEL} (DisCIPL-SMC)",
    "disciple_importance": f"{HUGGINGFACE_MODEL} (DisCIPL-IS)",
    "disciple_rejection": f"{HUGGINGFACE_MODEL} (DisCIPL-RS)",
    "disciple_iid": f"{HUGGINGFACE_MODEL} (DisCIPL-IS)",
    # "unconditional_sampling": f"Unconditional Sampling",
    "huggingface": f"{HUGGINGFACE_MODEL} (Sampling)",
    "huggingface_beam_search": f"{HUGGINGFACE_MODEL} (Beam Search)",
    "gpt-4o-mini": f"GPT-4o-mini",
    "gpt-4o": f"GPT-4o",
    "o1": "o1",
}

# These pipelines were only run for N=1
SINGLETON_PIPELINES = ["o1"]

PALETTE = {
    # PIPELINES["disciple"]: "tab:orange",
    PIPELINES["disciple_smc"]: "tab:orange",
    PIPELINES["disciple_importance"]: "tab:green",
    PIPELINES["disciple_rejection"]: "tab:red",
    PIPELINES["disciple_iid"]: "tab:green",
    # PIPELINES["unconditional_sampling"]: "tab:blue",
    PIPELINES["huggingface"]: "tab:pink",
    PIPELINES["huggingface_beam_search"]: "tab:purple",
    PIPELINES["gpt-4o-mini"]: "tab:cyan",
    PIPELINES["gpt-4o"]: "tab:blue",
    PIPELINES["o1"]: "tab:gray",
}

SAVE_FIGS = True
if SAVE_FIGS:
    FIGS_PATH = os.path.join("figures", DATASET, SUBTASKS).lower().replace(" ", "_").replace("-", "_")
    os.makedirs(FIGS_PATH, exist_ok=True)

In [ ]:
# load the data
df_list = []
existing_pipelines = set()
for timestamp, pipelines in TIMESTAMP_TO_PIPELINES.items():
    if pipelines is None:
        pipelines = PIPELINES.keys() # default to all pipelines

    print(f"Loading results from {timestamp}: {pipelines}")
    _df = pd.read_json(f"results/{timestamp}/results.json")

    # warn on duplicate pipelines
    new_pipelines = set(_df["pipeline_name"].unique())
    duplicates = new_pipelines.intersection(existing_pipelines)
    if duplicates:
        print(f"WARNING: {duplicates} already in existing pipelines")

    # filter the pipelines
    _df = _df[_df["pipeline_name"].isin(pipelines)]
    df_list.append(_df)
df = pd.concat(df_list).reset_index(drop=True)

In [ ]:
# Unpack task dict into columns
print("Unpacking task dict into columns")
df_task = df["task"].apply(pd.Series)
# Rename columns
df_task = df_task.rename(columns={"prompt": "task_prompt"})
# Drop columns
if "evaluators" in df_task.columns:
    df_task = df_task.drop(columns=["evaluators"])
if "examples" in df_task.columns:
    df_task = df_task.drop(columns=["examples"])

# Unpack evaluate_results dict into columns
print("Unpacking evaluate_results dict into columns")
df_evaluate_results = df["evaluate_results"].apply(pd.Series)
# Rename columns
# df_evaluate_results = df_evaluate_results.rename(columns={metric: f"eval_{metric}" for metric in df_evaluate_results.columns})

# Combine dataframes
print("Combining dataframes")
df = pd.concat(
    [
        df_task, df, df_evaluate_results
    ],
    axis=1,
)
df = df.drop(columns=["task", "evaluate_results"])

if "valid" in df.columns:
    df["valid_true"] = df["valid"] == True
elif "strict" in df.columns:
    df["valid_true"] = df["strict"] == True
else:
    raise ValueError("No `valid` column found")

# df["error"] = df["error"].astype(str)
if "error" not in df.columns:
    df["error"] = None
df["error_true"] = ~df["error"].isnull()

# Convert task_types (list) to string
print("Converting task_types (list) to string")
df["task_type"] = df["task_types"].apply(lambda x: ", ".join(x))

# Extract task_type_n from task_type
df["task_type_n"] = df["task_type"].str.extract(r"(\d+)").astype(float)

# Map task_type to task_level
print("Mapping task_type to task_level")
def task_type_to_level(task_type):
    if "sent" in task_type:
        return "sentence"
    elif "para" in task_type:
        return "paragraph"
    else:
        return "other"
df["task_level"] = df["task_type"].apply(task_type_to_level)

df["method"] = df["pipeline_name"].map(PIPELINES)
df["method"] = pd.Categorical(df["method"], categories=[v for k, v in PIPELINES.items() if k in df["pipeline_name"].unique()])

# Reorder columns (put "task_id", "task_types", "task_prompt" first, then the rest)
COL_ORDER = ["task_id", "task_types", "task_prompt", "pipeline_name", "method", "N"]
COL_ORDER += [col for col in df.columns if col not in COL_ORDER]
df = df[COL_ORDER]

# Filter out pipelines that are not in the list
df = df[df["pipeline_name"].isin(PIPELINES.keys())].reset_index(drop=True)

# For each singleton pipeline, duplicate the data for each unique value of N
for pipeline in SINGLETON_PIPELINES:
    if pipeline in df["pipeline_name"].values:
        singleton_df = df[df["pipeline_name"] == pipeline]
        for n in df["N"].unique():
            if n not in singleton_df["N"].values:
                new_rows = singleton_df.copy()
                new_rows["N"] = n
                df = pd.concat([df] + [new_rows] * n, ignore_index=True)

df["singleton"] = df["pipeline_name"].isin(SINGLETON_PIPELINES)

# Only include results from final attempts
df_all_attempts = df.copy()
df = df[df["is_final_attempt"] == True].reset_index(drop=True)

In [ ]:
df.columns

## Data validation checks

In [ ]:
N_TASKS = df["task_id"].nunique()
print(f"Number of tasks: {N_TASKS}")

assert not df["N"].isnull().any()

pipeline_counts = df[
    "pipeline_name"
].value_counts()

if pipeline_counts.nunique() != 1:
    raise ValueError(f"Methods have different number of rows: {pipeline_counts.to_dict()}")

for N, group in df.groupby("N"):
    n_tasks_per_method = group["pipeline_name"].value_counts() / N
    if (n_tasks_per_method != N_TASKS).all():
        raise ValueError(f"Number of tasks is not consistent for N={N}")

In [ ]:
n_tasks_per_method

In [ ]:
n_tasks_per_method

In [ ]:
df.groupby(["N", "method"]).size().unstack().plot(kind="bar", stacked=False, color=[PALETTE[method] for method in df["method"].unique()])

In [ ]:
df.groupby(["method"])["error_true"].value_counts(normalize=True)

# Eval metrics

In [ ]:
df["valid_breakdown"] = df[["valid_true", "error_true"]].apply(
    lambda x: (
        "Error" if x["error_true"] else ("Valid" if x["valid_true"] else "Invalid")
    ),
    axis=1,
)

df["N_value"] = df["N"].astype(str)

palette = {
    "Valid": "tab:green",
    "Invalid": "tab:red",
    "Error": "tab:grey",
}

g = sns.displot(
    data=df,
    # x="N_value",
    x="task_type",
    hue="valid_breakdown",
    col="method",
    stat="proportion",
    discrete=True,
    multiple="fill",
    palette=palette,
    shrink=0.9,
    hue_order=list(palette.keys()),
)

for ax in g.axes.flat:
    ax.set_title(ax.get_title().split(" = ")[-1])

In [ ]:
sns.catplot(kind="bar", col="task_type", data=df, x="N", y="valid_true", hue="method", palette=PALETTE)

In [ ]:
g = sns.FacetGrid(
    df,
    col="task_type",
    # col="task_type_n",
    # row="task_level",
    hue="method",
    palette=PALETTE,
    aspect=1.0,
    sharex=False,
)

g.map(sns.lineplot, "N", "valid_true", marker="o")
g.add_legend(title="Method")

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    ax.set_title(
        ax.get_title().split(" = ")[-1],
    )

    # Set y-axis limits
    # ax.set_ylim(None, 1)

g.set_axis_labels("Samples")
g.set_ylabels("Valid")

In [ ]:
df[["method", "valid_true", "check_result", "error_true", "text"]]

In [ ]:
# Get the ordered categories from the method column
categories = df["method"].cat.categories

# Create subplots: arrange them in a row
fig, axs = plt.subplots(1, len(categories), figsize=(4 * len(categories), 4))

# In case there's only one category, wrap axs in a list
if len(categories) == 1:
    axs = [axs]

for ax, method in zip(axs, categories):
    # Filter the dataframe for the current method
    df_method = df[df["method"] == method]

    # Compute the confusion matrix using valid_true and check_result
    cm = confusion_matrix(
        y_true=df_method["valid_true"],
        y_pred=df_method["check_result"],
        labels=[False, True],
    )

    # Display the confusion matrix on the current axis
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=["Invalid", "Valid"]
    )
    disp.plot(ax=ax, colorbar=False)

    # Set a title for the plot
    ax.set_title(method)

plt.tight_layout()
plt.show()

## Pass@k

In [ ]:
def compute_pass_at_k(df, k_values=range(1, 11)):
    """For each group of size k, pass@k is True if any rows in the group are valid and False otherwise."""
    GROUP_COLS = ["task_type", "task_id", "method", "N"]
    df_list = []
    for (task_type, task_id, method, N), group in df.groupby(GROUP_COLS)[
        df.columns.tolist()
    ]:
        for k in k_values:
            # if len(group) < k:
            #     break
            if N < k:
                continue

            if group["error_true"].all():
                pass_at_k = False
                error_true = True
            else:
                pass_at_k = group.nlargest(k, "weight", keep="all")["valid_true"].any() # weighted pass@k
                # pass_at_k = group.sample(n=k, replace=False)["valid_true"].any()  # random pass@k
                error_true = False

            singleton = group["singleton"].unique()
            if len(singleton) > 1:
                raise ValueError(f"More than one singleton value: {singleton}")
            singleton = singleton[0]

            task_level = group["task_level"].unique()
            if len(task_level) > 1:
                raise ValueError(f"More than one task_level value: {task_level}")
            task_level = task_level[0]


            df_list.append(
                {
                    "task_id": task_id,
                    "task_type": task_type,
                    "task_level": task_level,
                    "method": method,
                    "N": N,
                    "k": k,
                    f"pass@k": pass_at_k,
                    "error_true": error_true,
                    "singleton": singleton,
                }
            )
    return pd.DataFrame(df_list)


df_pass_at_k = compute_pass_at_k(df, k_values=df["N"].unique().tolist())
df_pass_at_k

In [ ]:
df_pass_at_k_counts = df_pass_at_k.groupby(["method", "N", "k"]).count()

_df_missing_tasks = df_pass_at_k_counts[df_pass_at_k_counts.ne(N_TASKS).any(axis=1)]
assert _df_missing_tasks.empty, _df_missing_tasks

print(f"Each row has {N_TASKS} tasks.")

### pass@1

In [ ]:
df_pass_at_1 = df_pass_at_k[df_pass_at_k["k"] == 1]
df_pass_at_1

In [ ]:
ax = sns.lineplot(
    data=df_pass_at_1,
    x="N",
    y="pass@k",
    hue="method",
    palette=PALETTE,
    # marker="o"
)

sns.despine(ax=ax, offset=10)

# Set x-axis labels to correspond exactly to the values of the points
x_values = df["N"].unique()
# ax.set_xscale("log", base=2)
ax.set_xticks(x_values)
ax.set_xticklabels(x_values)

plt.xlabel("Samples")
plt.ylabel("Pass@1")
plt.title(f"Validity for {DATASET} {SUBTASKS}")

# Move the legend outside the plot
ax.legend(title="Method", loc="center left", bbox_to_anchor=(1, 0.5))

In [ ]:
g = sns.FacetGrid(
    df_pass_at_1, col="task_type", hue="method", palette=PALETTE, aspect=1.0
)

g.map(sns.lineplot,
      "N",
      "pass@k",
    #   marker="o"
      )
g.add_legend(title="Method")

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    ax.set_title(
        ax.get_title().split(" = ")[-1],
    )

    # Set y-axis limits
    # ax.set_ylim(None, 1)

g.set_axis_labels("Samples")
g.set_ylabels(f"Pass@1")

In [ ]:
g = sns.FacetGrid(
    df_pass_at_1, col="task_type", hue="method", palette=PALETTE, aspect=1.0
)

g.map(sns.lineplot, "N", "error_true", marker="o")
# g.map(sns.barplot, "N", "error_true", order=sorted(df_pass_at_1["N"].unique()))
g.add_legend(title="Method")

# # Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:

    ax.set_title(
        ax.get_title().split(" = ")[-1],
    )

    # Set the x axis to df_pass_at_1["N"].unique()
    x_values = df_pass_at_1["N"].unique()
    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    ax.set_ylim(None, 1)

g.set_axis_labels("Samples")
g.set_ylabels(f"Runtime Error @ 1")

### Pass @k=N

In [ ]:
df_pass_at_k_equals_n = df_pass_at_k[df_pass_at_k["k"] == df_pass_at_k["N"]]

In [ ]:
# Overall plot

ax = sns.lineplot(
    data=df_pass_at_k_equals_n,
    x="N",
    y="pass@k",
    hue="method",
    palette=PALETTE,
    # marker="o"
)

sns.despine(ax=ax, offset=10)

# Set x-axis labels to correspond exactly to the values of the points
x_values = df["N"].unique()
# ax.set_xscale("log", base=2)
ax.set_xticks(x_values)
ax.set_xticklabels(x_values)

plt.xlabel("Samples")
plt.ylabel("Pass@k")
plt.title(f"Validity for {DATASET} {SUBTASKS}")

# Move the legend outside the plot
ax.legend(title="Method", loc="center left", bbox_to_anchor=(1, 0.5))

In [ ]:
# Task-level plot

g = sns.FacetGrid(
    df_pass_at_k_equals_n, col="task_type", hue="method", palette=PALETTE, aspect=1.0
)

g.map(
    sns.lineplot,
    "N",
    "pass@k",
    #   marker="o"
)
g.add_legend(title="Method")

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    ax.set_title(
        ax.get_title().split(" = ")[-1],
    )

    # Set y-axis limits
    # ax.set_ylim(None, 1)

g.set_axis_labels("Samples")
g.set_ylabels(f"Pass@k")

In [ ]:
from evaluations.plotting import plot_pass_at_k

fig = plot_pass_at_k(
    df_pass_at_k,
    DATASET,
    SUBTASKS,
    PALETTE,
    dash_methods = SINGLETON_PIPELINES,
    height_ratios=[2, 1],
)
plt.show()

if SAVE_FIGS:
    fig.savefig(os.path.join(FIGS_PATH, f"pass_at_k.pdf"), bbox_inches="tight", dpi=300)

### Pass@k

In [ ]:
g = sns.relplot(
    data=df_pass_at_k,
    x="N",
    y="pass@k",
    row="k",
    style="k",
    markers=False,
    dashes=False,
    hue="method",
    col="task_type",
    kind="line",
    palette=PALETTE,
    errorbar=None,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Remove "task_name = " from the subtitle
    ax.set_title(ax.get_title().replace("task_type = ", ""))

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k,
    x="N",
    y="pass@k",
    # row="task_type",
    col="k",
    style="k",
    markers=True,
    dashes=False,
    hue="method",
    kind="line",
    palette=PALETTE,
    errorbar=None,
    facet_kws={"sharex": False},
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df["N"].unique()

    ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k,
    x="k",
    y="pass@k",
    # row="task_type",
    # col="k",
    style="N",
    # size="N",
    dashes=False,
    hue="method",
    kind="line",
    palette=PALETTE,
    errorbar=None,
    markers=True,
    facet_kws={"sharex": False},
    aspect=1.5,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df_pass_at_k["k"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k,
    x="N",
    y="pass@k",
    # row="task_type",
    # col="k",
    style="k",
    size="k",
    dashes=False,
    hue="method",
    kind="line",
    palette=PALETTE,
    errorbar=None,
    markers=True,
    facet_kws={"sharex": False},
    aspect=1.5,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df_pass_at_k["N"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

In [ ]:
g = sns.relplot(
    data=df_pass_at_k[df_pass_at_k["N"] == df_pass_at_k["k"]],
    x="k",
    y="pass@k",
    hue="method",
    kind="line",
    palette=PALETTE,
    markers=True,
    facet_kws={"sharex": False},
    aspect=1.5,
)

# Set x-axis labels to correspond exactly to the values of the points
for ax in g.axes.flat:
    x_values = df_pass_at_k["k"].unique()

    # ax.set_xscale("log", base=2)
    ax.set_xticks(x_values)
    ax.set_xticklabels(x_values)

    # Set y-axis limits
    # ax.set_ylim(None, 1)

plt.xlabel("Samples")
plt.ylabel("Pass@k")
plt.title(f"Validity for {DATASET} {SUBTASKS}")

# Set the legend title
g.legend.set_title("Method")

In [ ]:
g = sns.barplot(
    data=df_pass_at_k[(df_pass_at_k["N"] == df_pass_at_k["k"]) & (df_pass_at_k["method"].str.contains("DisCIPL"))],
    x="k",
    y="error_true",
    hue="method",
    palette=PALETTE,
)

# Move the legend outside the plot
g.legend(title="Method", loc="center left", bbox_to_anchor=(1, 0.5))

sns.despine()

plt.xlabel("Samples")
plt.ylabel("Error Rate")
plt.title("Error Rate vs. Samples")

In [ ]:
k_max = df["N"].max()

df_pass_at_k_max = df_pass_at_k[df_pass_at_k["k"] == k_max]

g = sns.barplot(
    data=df_pass_at_k_max,
    x="task_type",
    y="pass@k",
    hue="method",
    palette=PALETTE,
)

# Move the legend outside the plot
g.legend(title="Method", loc='center left', bbox_to_anchor=(1, 0.5))

g.set_ylabel(f"Pass@{k_max}")

## Tables

In [ ]:
df_table = df_pass_at_k[df_pass_at_k["k"] == 1]
df_table = df_table[
    ((df_table["task_level"] == "sentence") & (df_table["N"] == 32)) |
    ((df_table["task_level"] == "paragraph") & (df_table["N"] == 8))
]

df_table = df_table.groupby(["method", "task_level"])[["pass@k", "error_true"]].agg("mean")
df_table = df_table.unstack("task_level")
df_table = df_table.swaplevel(0, 1, axis=1)
df_table = df_table.sort_index(axis=1, level=0)

# Force the column order so that for each task_level, "pass@k" comes before "error_true"
# Also, ensure that "sentence" appears left of "paragraph"
desired_order = [
    ("sentence", "pass@k"), ("sentence", "error_true"),
    ("paragraph", "pass@k"), ("paragraph", "error_true")
]
df_table = df_table.reindex(columns=desired_order)

df_table = df_table.rename(columns={"sentence": "Sentence", "paragraph": "Paragraph"}, level=0)
df_table = df_table.rename(columns={"pass@k": "Pass@1", "error_true": "Error"}, level=1)

# Reorder the rows to use the order defined in PIPELINES.values()
row_order = [PIPELINES[k] for k in PIPELINES if k in df["pipeline_name"].unique()]
df_table = df_table.reindex(row_order)

df_table = df_table.round(3)
df_table.index.name = None

df_table.to_latex(os.path.join(FIGS_PATH, "pass_at_k.tex"), float_format="%.3f")

df_table

# Error analysis

In [ ]:
df.groupby(["method"])["error_true"].value_counts(normalize=True)

In [ ]:
df["error"].isnull().value_counts(normalize=True)

In [ ]:
# PIPELINE_NAME = "disciple_iid"
PIPELINE_NAME = "disciple_smc"

In [ ]:
rows_per_task_type = (
    df[df["pipeline_name"] == PIPELINE_NAME]
    .groupby("task_type")["task_id"]
    .count()
    .to_frame()
)
rows_per_task_type

In [ ]:
error_counts = (
    df[df["pipeline_name"] == PIPELINE_NAME].copy()
    .groupby("task_type")["error"]
    .value_counts(dropna=False, normalize=False, ascending=False)
    .to_frame(name="count")
)

# Replace NaN with "No error"
error_counts = error_counts.reset_index()
error_counts["error"] = error_counts["error"].fillna("No error")
error_counts = error_counts.set_index(["task_type", "error"])

# Join with rows_per_task_type
error_counts = error_counts.join(rows_per_task_type, on="task_type")

# Divide "count" by "rows_per_task_type"
error_counts["count"] = error_counts["count"] / error_counts["task_id"]

# Drop the "task_id" column
error_counts = error_counts.drop(columns=["task_id"])

error_counts

In [ ]:
TOP_N_ERRORS = 20

# df_pipeline = df[df["pipeline_name"] == PIPELINE_NAME].copy()
df_pipeline = df_all_attempts[df_all_attempts["pipeline_name"] == PIPELINE_NAME].copy()
df_pipeline.loc[:, "error"] = df_pipeline.loc[:, "error"].fillna("No error")

# Get the top N most frequent errors
n_unique_errors = df_pipeline["error"].nunique()
top_errors = df_pipeline["error"].value_counts().nlargest(min(TOP_N_ERRORS - 1, n_unique_errors)).index

# Map all other errors to "Other"
df_pipeline.loc[~df_pipeline["error"].isin(top_errors), "error"] = "Other"

# Order the hue by error frequency
error_order = df_pipeline["error"].value_counts().index

with sns.plotting_context("talk", font_scale=0.75):
    g = sns.displot(
        data=df_pipeline,
        x="task_type",
        hue="error",
        hue_order=error_order,
        stat="proportion",
        discrete=True,
        multiple="fill",
        shrink=0.8,
        height=6,
        palette="tab20",
    )

plt.xlabel("")
plt.xticks(rotation=45)
plt.ylabel("Error Rate")

legend = g.legend
legend.set_title("")

if SAVE_FIGS:
    g.savefig(os.path.join(FIGS_PATH, "error_distribution.pdf"), bbox_inches="tight", dpi=300)

In [ ]:
df_error_wide = (
    df[df["pipeline_name"] == PIPELINE_NAME]
    .groupby("task_type")["error"]
    .value_counts(dropna=False, normalize=False, ascending=True)
    .reset_index()
    .set_index("error")
    .pivot(columns="task_type", values="count")
)
df_error_wide = df_error_wide.sort_index(ascending=False)

g = df_error_wide.plot(kind="barh", stacked=True, legend=False)
g.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
df_error = df[~df.error.isna()]

In [ ]:
plot = sns.countplot(data=df_error, x="task_type")

In [ ]:
df_all_attempts["valid_breakdown"] = df_all_attempts[
    ["valid_true", "error_true"]
].apply(
    lambda x: (
        "Error" if x["error_true"] else ("Valid" if x["valid_true"] else "Invalid")
    ),
    axis=1,
)

palette = {
    "Valid": "tab:green",
    "Invalid": "tab:red",
    "Error": "tab:grey",
}

with sns.plotting_context("talk", font_scale=1.):
    g = sns.displot(
        data=df_all_attempts[df_all_attempts["pipeline_name"] == "disciple_smc"],
        x="attempt",
        hue="valid_breakdown",
        stat="proportion",
        discrete=True,
        multiple="fill",
        palette=palette,
        shrink=0.9,
        hue_order=list(palette.keys()),
    )

for ax in g.axes.flat:
    ax.set_title(ax.get_title().split(" = ")[-1])

g._legend.set_title("")
g._legend.set_bbox_to_anchor((0.9, 0.5))
plt.xlabel("Retry #");



if SAVE_FIGS:
    g.savefig(os.path.join(FIGS_PATH, "validity_over_attempts.pdf"), bbox_inches="tight", dpi=300)

# Evaluation of text

In [ ]:
df["text_length"] = df["text"].apply(lambda x: len(x) if x is not None else 0)
sns.displot(data=df, x="text_length", hue="method", kind="kde", palette=PALETTE).set(
    xlim=(0, 1000)
)

In [ ]:
df.groupby("method")["text_length"].describe()

In [ ]:
N = 3
RANDOM_STATE = 888
RANDOM_TASK_IDS = (
    df.groupby("task_type").sample(n=N, random_state=RANDOM_STATE)["task_id"].tolist()
)
print(RANDOM_TASK_IDS)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    # display(
    #     df.groupby(["task_id"]).sample(n=3, random_state=0)[
    #         ["task_type", "task_id", "method", "task_prompt", "text"]
    #     ]
    # )
    display(
        df[
            df["task_id"].isin(RANDOM_TASK_IDS) & (df["N"] == df["N"].max())
        ].sort_values(
            ["task_type", "task_id", "method", "weight"],
            ascending=[True, True, True, False],
        )[
            [
                "task_type",
                "task_id",
                "method",
                "task_prompt",
                "text",
                "text_length",
                "weight",
                "valid_true",
            ]
        ]
    )

# Debugging